# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassanbuilds/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I will use a Random Forest classifier for this modeling lane.

The goal is to identify content associated with a declining trend so that pages can be prioritized for refresh review. Random Forest is suitable because it can model nonlinear relationships between content, search, and traffic signals without requiring strong assumptions about their relationships.

It also provides feature importance measures that give a directional view of which observable signals the model relies on.

I will compare the Random Forest with the Week-4 rule-based baseline using the same held-out test set and the same ranking metrics. The goal is to measure whether the model provides a useful improvement over the simpler baseline rather than choosing a more complex method simply because it is more complex.

In [2]:
!pip -q install datasets

from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset(
    "FlyRank/internship-starter",
    split="train"
)

df = dataset.to_pandas()

print("Dataset shape:", df.shape)
display(df.head())

# Create the target used for supervised modeling.
# 1 = declining, 0 = not declining.

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget percentage:")
print(
    (df["is_declining_label"].value_counts(normalize=True) * 100)
    .round(2)
)

README.md:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

content_refresh_anonymized.csv:   0%|          | 0.00/8.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30000 [00:00<?, ? examples/s]

Dataset shape: (30000, 53)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False



Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentage:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I will use the official client-aware holdout split.

The dataset contains multiple content rows for each client. A random row-level split could place pages from the same client in both training and test data, which could make evaluation less representative of performance on an unseen client.

The official modeling script therefore holds out approximately 20% of clients for testing when the data supports a valid two-class split. This keeps the clients in the test set separate from the clients used for training.

The Week-4 baseline will be evaluated on these exact same test rows so that the model and baseline comparison is fair.*italicized text*

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I will train the official modeling toolkit's Logistic Regression, Decision Tree, and Random Forest models using the official client-aware split.

The Random Forest is the main model for this lane because it provides a strong ranking signal and feature importance values.

I will reproduce the Week-4 rule-based baseline on the exact same held-out test rows. The models and baseline will be compared using Precision@20, Precision@50, and Precision@100.

This keeps the comparison on the same data, split, and metrics and allows the model to be judged against the simpler baseline.

In [4]:
import subprocess
import sys
from pathlib import Path

# Clone the official FlyRank reference repository
repo = Path("/content/flyrank-reference")

if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
            str(repo)
        ],
        check=True
    )

# Make the official scripts importable.
# This fixes the "ModuleNotFoundError: No module named 'ml_utils'" error.
scripts_path = str(repo / "scripts")

if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

print("Official FlyRank repository ready.")
print("Scripts path:", scripts_path)

Official FlyRank repository ready.
Scripts path: /content/flyrank-reference/scripts


In [5]:
# Prepare the official feature vector
subprocess.run(
    ["python", "scripts/01_prepare_features.py"],
    cwd=repo,
    check=True
)

# Create the official Week-4 baseline scores
subprocess.run(
    ["python", "scripts/02_baseline_score.py"],
    cwd=repo,
    check=True
)

print("Official feature preparation and baseline creation completed.")

Official feature preparation and baseline creation completed.


In [6]:
feature_path = (
    repo / "data" / "processed" / "refresh_feature_vector.csv"
)

baseline_path = (
    repo / "data" / "processed" / "baseline_refresh_queue.csv"
)

features_df = pd.read_csv(feature_path)
baseline_df = pd.read_csv(baseline_path)

print("Feature vector shape:", features_df.shape)
print("Baseline shape:", baseline_df.shape)

Feature vector shape: (30000, 52)
Baseline shape: (30000, 22)


In [7]:
import importlib.util

script_path = repo / "scripts" / "03_train_model.py"

spec = importlib.util.spec_from_file_location(
    "flyrank_train_model",
    script_path
)

flyrank_model = importlib.util.module_from_spec(spec)

sys.modules["flyrank_train_model"] = flyrank_model

spec.loader.exec_module(flyrank_model)

print("Official FlyRank modeling utilities loaded.")

Official FlyRank modeling utilities loaded.


In [8]:
scripts_path = str(repo / "scripts")

if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

In [9]:
# Load the official processed data
frame = pd.read_csv(feature_path)
baseline_frame = pd.read_csv(baseline_path)

print("Rows:", len(frame))
print("Features:", len(frame.columns))
print(
    "Target positive rows:",
    int(frame["is_declining_label"].sum())
)
print(
    "Target positive rate:",
    round(frame["is_declining_label"].mean(), 4)
)

# Build the official feature matrix
feature_frame, feature_columns = (
    flyrank_model.build_feature_matrix(frame)
)

# Target
target_series = frame["is_declining_label"].astype(int)

# Official client-aware split
train_indices, test_indices, split_strategy = (
    flyrank_model.make_client_aware_split(
        frame,
        target_series
    )
)

print("\nSplit strategy:", split_strategy)
print("Training rows:", len(train_indices))
print("Test rows:", len(test_indices))

# Train/test data
train_features = feature_frame.iloc[train_indices]
test_features = feature_frame.iloc[test_indices]

train_target = target_series.iloc[train_indices]
test_target = target_series.iloc[test_indices]

# Baseline scores on the exact same test rows
baseline_lookup = baseline_frame.set_index(
    "content_id"
)["baseline_refresh_score"]

baseline_test_scores = (
    frame.iloc[test_indices]["content_id"]
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)

# Baseline metrics
baseline_metrics = flyrank_model.metric_payload(
    test_target,
    baseline_test_scores,
    prefix="baseline_"
)

# Build and train official models
models = flyrank_model.build_models()

model_results = {}
trained_models = {}

for model_name, model in models.items():

    model.fit(
        train_features,
        train_target
    )

    probabilities = flyrank_model.predict_probability(
        model,
        test_features
    )

    model_results[model_name] = (
        flyrank_model.metric_payload(
            test_target,
            probabilities
        )
    )

    trained_models[model_name] = model


# Build comparison table
results_rows = [
    {
        "Method": "Week-4 baseline",
        "Precision@20": baseline_metrics["baseline_precision_at_20"],
        "Precision@50": baseline_metrics["baseline_precision_at_50"],
        "Precision@100": baseline_metrics["baseline_precision_at_100"],
    }
]

for model_name, metrics in model_results.items():

    results_rows.append({
        "Method": model_name.replace("_", " ").title(),
        "Precision@20": metrics["precision_at_20"],
        "Precision@50": metrics["precision_at_50"],
        "Precision@100": metrics["precision_at_100"],
    })

comparison = pd.DataFrame(results_rows)

for column in [
    "Precision@20",
    "Precision@50",
    "Precision@100"
]:
    comparison[column] = comparison[column].round(3)

print("\nModel vs baseline:")
display(comparison)

Rows: 30000
Features: 52
Target positive rows: 16262
Target positive rate: 0.5421

Split strategy: client_holdout
Training rows: 27675
Test rows: 2325

Model vs baseline:


,Method,Precision@20,Precision@50,Precision@100
0,Week-4 baseline,0.15,0.24,0.36
1,Logistic Regression,0.35,0.40,0.44
2,Decision Tree,0.45,0.58,0.62
3,Random Forest,0.65,0.74,0.72


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest produced false positives and false negatives on the held-out test set when using a probability threshold of 0.5. This shows that the model is useful for ranking but does not perfectly classify every page.

The highest-confidence false positives show cases where the model assigned a relatively high probability to pages that were not labeled as declining. The false negatives show the opposite pattern: some declining pages received relatively low model probabilities.

The most important Random Forest features were `days_with_impressions`, `log_impressions_90d`, `avg_position`, and `content_age_days`. These importance values provide a directional view of which observable signals the model relied on. They should not be interpreted as causal effects.

Overall, the Random Forest achieved a Precision@50 of 0.74 compared with 0.24 for the Week-4 baseline on the held-out test set. This measured improvement suggests that the model provides a stronger ranking signal for prioritizing potentially declining content.

I would use the model as a decision-support tool rather than as an automatic decision-maker.

In [10]:
# Get Random Forest predictions
rf_probabilities = flyrank_model.predict_probability(
    trained_models["random_forest"],
    test_features
)

error_analysis = frame.iloc[test_indices][
    ["content_id", "client_id", "is_declining_label"]
].copy()

error_analysis["probability"] = rf_probabilities

# Classify predictions using a 0.5 probability threshold
error_analysis["predicted_label"] = (
    error_analysis["probability"] >= 0.5
).astype(int)

# Identify false positives and false negatives
false_positives = error_analysis[
    (error_analysis["predicted_label"] == 1)
    & (error_analysis["is_declining_label"] == 0)
].copy()

false_negatives = error_analysis[
    (error_analysis["predicted_label"] == 0)
    & (error_analysis["is_declining_label"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nHighest-confidence false positives:")
display(
    false_positives
    .sort_values("probability", ascending=False)
    .head(10)
)

print("\nHighest-confidence false negatives:")
display(
    false_negatives
    .sort_values("probability", ascending=True)
    .head(10)
)

False positives: 529
False negatives: 233

Highest-confidence false positives:


,content_id,client_id,is_declining_label,probability,predicted_label
23250,content_d2dffcc697a4,client_f74efabef1,0,0.737130,1
23559,content_00603b0349b4,client_f74efabef1,0,0.734944,1
25913,content_331182ca4cae,client_f74efabef1,0,0.733631,1
23750,content_e55b8ab078b0,client_f74efabef1,0,0.733059,1
10155,content_643f585dc7f7,client_f74efabef1,0,0.731120,1
5966,content_f5013794ba57,client_f74efabef1,0,0.730532,1
28337,content_ea4417d89e2c,client_f74efabef1,0,0.729056,1
4249,content_db1cd41b4b4f,client_f74efabef1,0,0.729052,1
21530,content_b15a8dbdf66f,client_f74efabef1,0,0.727853,1
2380,content_96da95476e63,client_f74efabef1,0,0.724807,1



Highest-confidence false negatives:


,content_id,client_id,is_declining_label,probability,predicted_label
5770,content_28b4223f4e5f,client_98a3ab7c34,1,0.079867,0
3879,content_34b14c00f80c,client_d4735e3a26,1,0.082196,0
27177,content_79ac977c6e0b,client_f74efabef1,1,0.149546,0
22991,content_472ce7ae14c0,client_d4735e3a26,1,0.152184,0
5608,content_a55d958ec725,client_d4735e3a26,1,0.163987,0
12864,content_f1ef151d5e36,client_d4735e3a26,1,0.165946,0
12076,content_230de4c50860,client_d4735e3a26,1,0.169816,0
25838,content_cbc3b52a2ac1,client_98a3ab7c34,1,0.171345,0
13659,content_4c437dd8c1ee,client_d4735e3a26,1,0.174066,0
23810,content_37804210415c,client_d4735e3a26,1,0.174828,0


In [11]:
# Random Forest feature importance

rf_importance = flyrank_model.top_feature_importance(
    trained_models["random_forest"],
    feature_columns,
    limit=10
)

importance_df = pd.DataFrame(rf_importance)

print("Top 10 Random Forest features:")
display(importance_df)

Top 10 Random Forest features:


,feature,importance
0,days_with_impressions,0.134951
1,log_impressions_90d,0.129377
2,avg_position,0.109203
3,content_age_days,0.092048
4,char_count,0.038676
5,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
7,word_count,0.035406
8,ctr,0.035156
9,scroll_rate,0.033876


The feature importance results provide a directional view of the signals used by the Random Forest. The most important features are not interpreted as causal drivers of decline. Instead, they indicate which available features contributed most to the model's predictions.

The error review is also useful because a high model score does not guarantee that a page is actually declining. False positives represent pages prioritized by the model that were not labeled as declining, while false negatives represent declining pages that the model did not prioritize.

Overall, the measured improvement over the baseline supports using the Random Forest as a decision-support ranking tool, while the remaining errors show that model predictions should not be treated as perfect classifications.

## Self-check

Before submitting, I confirmed:

- [x] Every section above is filled with markdown and supporting code.
- [x] The notebook runs top to bottom without errors.
- [x] No client names, URLs, or private queries are included in the analysis.
- [x] Claims use careful words such as measured, observed, directional, and decision-support.
- [x] The notebook is committed under `work/notebooks/`.